# Colorization

**Owner:** Ashok

DDColor (Kang et al., 2023) on grayscale and sepia photos; skipped when the photo already has color. Color error is measured with Delta E CIEDE2000 and LPIPS.

| Item | Detail |
|---|---|
| Stage | 4 of 5 in the pipeline (after repair, before enhancement) |
| Model | DDColor, pretrained from Hugging Face (`piddnad/ddcolor_*`), inference only |
| Input / output | Gray or sepia RGB image -> color RGB image |
| Skip rule | Photo already in color -> returned unchanged, skip is logged |
| Requirements | R2 (algorithm + theory), R4 (metrics), E1 (pretrained Hugging Face model) |

**Sections**
1. Setup
2. How DDColor works (theory)
3. Load the model
4. Skip rule: is the photo gray, sepia or color?
5. The `Colorizer` class (lives in `src/colorization.py`)
6. Test images
7. Results and metrics (Delta E, LPIPS)
8. Tests
9. Resource smoke test (peak memory, seconds per image)
10. Findings and limits

> Colorization is a plausible guess, not a recovery of the true colors. Every output from this stage is labelled that way in the demo and report.

## 1. Setup

In [ ]:
import os, sys, subprocess
from pathlib import Path

# On Colab: clone the repo. Locally: walk up from the working directory to the project root.
REPO_URL = "https://github.com/diaeshantony-usd/old-photo-restoration-studio"
if Path("/content").exists() and not Path("/content/old-photo-restoration-studio").exists():
    subprocess.run(["git", "clone", REPO_URL, "/content/old-photo-restoration-studio"], check=True)

def get_project_root() -> Path:
    cur = Path.cwd().resolve()
    for c in [cur, *cur.parents, Path("/content/old-photo-restoration-studio")]:
        if (c / "src").exists() and (c / "notebooks").exists():
            return c
    return cur

ROOT = get_project_root()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
DATA_DIR = ROOT / "data" / "cleaned"          # produced by notebook 01 (or unzipped from the team Drive)

DDCOLOR_DIR = "/content/DDColor" if Path("/content").exists() else str(ROOT.parent / "DDColor")
if not os.path.exists(DDCOLOR_DIR):
    subprocess.run(["git", "clone", "https://github.com/piddnad/DDColor.git", DDCOLOR_DIR], check=True)

# DDColor's own requirements pin old versions, so install only what inference needs.
!pip install -q lpips scikit-image timm einops huggingface_hub opencv-python-headless
print("project root:", ROOT)

In [ ]:
import time, random
import numpy as np
import cv2
import torch
import pandas as pd
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
OUT_DIR = "notebooks/04_colorization"      # small outputs the report needs are saved here
print("device:", DEVICE, "|", torch.cuda.get_device_name(0) if DEVICE == "cuda" else "no GPU (slow)")

## 2. How DDColor works

Colorization is ambiguous: a gray shirt could have been red, blue or green. Older methods predict one color per pixel from the gray input and tend to give washed-out, brownish results.

DDColor (Kang et al., 2023) works in **Lab color space**, where `L` is brightness and `a`, `b` are color. The gray photo supplies `L`, and the model only has to predict `a` and `b`. It has three parts:

1. **Backbone** (ConvNeXt) turns the gray image into multi-scale features.
2. **Pixel decoder** upsamples those features back to image resolution, so fine spatial detail is kept.
3. **Color decoder** is a transformer with a set of learned **color queries**. Each query learns to look for one kind of semantic color pattern (skin, sky, foliage) and attends to image features to decide *where* that color belongs. The output is combined with the pixel-decoder features to give the `a`, `b` channels.

Two decoders (one for spatial detail, one for semantic color) is what "dual decoders" means. The model is also trained with a **colorfulness loss** so results do not collapse to dull colors.

We use the **pretrained** weights and do no training here, which is why this stage fits easily on a free T4.

Available checkpoints (Hugging Face, `piddnad/`):

| Checkpoint | Notes |
|---|---|
| `ddcolor_paper` | Weights from the paper (ConvNeXt-L) |
| `ddcolor_modelscope` | Larger-data variant, usually the most robust on real photos |
| `ddcolor_artistic` | Trained on artistic images, more saturated |
| `ddcolor_paper_tiny` | Smaller (ConvNeXt-T), the memory fallback from the design doc |

## 3. Load the model

In [ ]:
from src.colorization import (Colorizer, load_ddcolor, classify_color, chroma_stats, is_grayscale,
                              to_gray, to_sepia, GRAY_CHROMA, SEPIA_CHROMA, SEPIA_HUE_STD)
from src.metrics import delta_e, lpips_distance, colorfulness

ddcolor = load_ddcolor(ddcolor_dir=DDCOLOR_DIR)     # piddnad/ddcolor_modelscope; "ddcolor_paper_tiny" is the memory fallback
print("DDColor loaded")

## 4. Skip rule: is the photo gray, sepia or color?

The pipeline must not colorize a photo that already has color (it would overwrite real information). We classify each photo from its **chroma** in Lab space, which is the distance from neutral gray:

- **gray:** almost no chroma anywhere
- **sepia:** a little chroma, but nearly all of it in one hue (brownish-yellow), because sepia is one tint applied to brightness
- **color:** otherwise

The thresholds are first guesses and are tuned on the test images in section 6.

In [ ]:
print(f"thresholds: gray < {GRAY_CHROMA} chroma; sepia < {SEPIA_CHROMA} chroma and hue spread < {SEPIA_HUE_STD} rad")

## 5. The `Colorizer` class

The class lives in `src/colorization.py` (imported above) so the notebook, the tests and the demo share one implementation. It returns the image plus a small record of what happened (`ran`, `kind`, `reason`), so the demo and the logs can say why a photo was or was not colorized.

In [ ]:
import logging
logging.basicConfig(level=logging.INFO, format="%(name)s: %(message)s")
colorizer = Colorizer(ddcolor)

## 6. Test images

The real damaged scans have no color ground truth, so to measure color error we start from **clean color photos**, strip the color (gray and sepia versions), colorize them, and compare with the original. This isolates the colorization stage from repair.

Source: the clean photos of OpenPhoto Restore in `data/cleaned/openphoto_restore/clean/` (made by notebook 01). The **last N files in sorted order** are the test images. If the folder is missing (no data on this machine yet), the notebook falls back to four public sample photos from scikit-image so every cell still runs.

In [ ]:
N_TEST = 20
MAX_SIDE = 512

def load_samples(n=N_TEST):
    clean_dir = DATA_DIR / "openphoto_restore" / "clean"
    files = sorted(clean_dir.glob("*.png"))[-n:] if clean_dir.exists() else []
    if files:
        imgs = {f.stem: cv2.cvtColor(cv2.imread(str(f)), cv2.COLOR_BGR2RGB) for f in files}
        source = "openphoto_restore/clean"
    else:
        from skimage import data
        imgs = {"astronaut": data.astronaut(), "chelsea": data.chelsea(), "coffee": data.coffee(), "rocket": data.rocket()}
        source = "skimage fallback (data/cleaned not found)"
    out = {}
    for k, v in imgs.items():
        sc = MAX_SIDE / max(v.shape[:2])
        out[k] = cv2.resize(v, (round(v.shape[1] * sc), round(v.shape[0] * sc)), interpolation=cv2.INTER_AREA)
    print(f"{len(out)} test images from {source}")
    return out

samples = load_samples()
first = next(iter(samples.values()))
show = list(samples.items())[:6]
fig, axs = plt.subplots(1, len(show), figsize=(3.2 * len(show), 3))
for ax, (name, im) in zip(np.atleast_1d(axs), show):
    ax.imshow(im); ax.set_title(name[:18], fontsize=8); ax.axis("off")
plt.show()

In [ ]:
# Check the skip-rule thresholds on every sample in all three forms.
print(f"{'image':16s} {'form':6s} {'chroma':>7s} {'hue_std':>8s}  class")
bad = 0
for name, im in samples.items():
    for form, img in (("color", im), ("gray", to_gray(im)), ("sepia", to_sepia(to_gray(im)))):
        c, h = chroma_stats(img); k = classify_color(img)
        flag = "" if k == form else "   <-- WRONG"
        bad += k != form
        print(f"{name:16s} {form:6s} {c:7.2f} {h:8.3f}  {k}{flag}")
print("\nmisclassified:", bad)

In [ ]:
# Real photos: how does the skip rule label the real damaged film scans (including faded color)?
film_dir = DATA_DIR / "film_scans_4k" / "damaged"
rows = []
for f in (sorted(film_dir.glob("*.png"))[:44] if film_dir.exists() else []):
    im = cv2.cvtColor(cv2.imread(str(f)), cv2.COLOR_BGR2RGB)
    im = cv2.resize(im, (512, round(512 * im.shape[0] / im.shape[1])), interpolation=cv2.INTER_AREA)
    c, h = chroma_stats(im)
    rows.append(dict(file=f.name, chroma=round(c, 2), hue_std=round(h, 3), label=classify_color(im)))
if rows:
    film = pd.DataFrame(rows)
    display(film["label"].value_counts().to_frame("scans"))
    display(film.sort_values("chroma").head(8))
    print("Check scans labelled color but with low chroma (faded prints); adjust the thresholds in src/colorization.py if they are wrong.")
else:
    print("film_scans_4k not found: run notebook 01 first (skipped).")

## 7. Results and metrics

| Metric | Measures | Better |
|---|---|---|
| Delta E (CIEDE2000) | Color difference to the original in Lab space (about 1 is barely visible, above 10 is clearly different) | Lower |
| LPIPS | How different the result looks to people, from deep features (Zhang et al., 2018) | Lower |

**How to read Delta E here:** there are many valid colorizations of a gray photo, so Delta E to the one true original is expected to stay well above zero. A **gray baseline** (leave the photo gray) gives the floor to beat, and we also report **colorfulness** (Hasler and Susstrunk, 2003) to show whether results are vivid or washed out.

In [ ]:
# delta_e, lpips_distance and colorfulness come from src/metrics.py (imported in section 3).

In [ ]:
rows, panels = [], []
for name, original in samples.items():
    for form, src in (("gray", to_gray(original)), ("sepia", to_sepia(to_gray(original)))):
        out = colorizer.colorize(src)
        rows.append(dict(
            image=name, input=form,
            dE_gray_baseline=delta_e(original, to_gray(original)),
            dE_ddcolor=delta_e(original, out),
            lpips_gray_baseline=lpips_distance(original, to_gray(original)),
            lpips_ddcolor=lpips_distance(original, out),
            colorfulness_original=colorfulness(original),
            colorfulness_ddcolor=colorfulness(out),
        ))
        panels.append((name, form, src, out, original))

df = pd.DataFrame(rows)
display(df.round(3))
print("\nmean over all images:")
display(df.drop(columns=["image", "input"]).mean().round(3).to_frame("mean").T)

In [ ]:
fig, axs = plt.subplots(len(panels), 3, figsize=(10, 3.2 * len(panels)))
for row, (name, form, src, out, original) in zip(axs, panels):
    for ax, im, t in zip(row, (src, out, original), (f"input ({form})", "DDColor", "original color")):
        ax.imshow(im); ax.set_title(f"{name} - {t}", fontsize=9); ax.axis("off")
plt.tight_layout()
fig.savefig(f"{OUT_DIR}/colorization_grid.png", dpi=120)
df.to_csv(f"{OUT_DIR}/colorization_metrics.csv", index=False)
plt.show()

## 8. Tests

The skip-rule and output-size tests that need no model already run as pytest cases in `tests/test_colorization.py`; these cells repeat them with the real DDColor model.

In [ ]:
def test_color_input_skips_colorization():
    im = first
    out = colorizer.colorize(im)
    assert colorizer.last_info["ran"] is False and colorizer.last_info["kind"] == "color"
    assert np.array_equal(out, im), "color photo must come back unchanged"

def test_gray_and_sepia_are_colorized():
    for src in (to_gray(first), to_sepia(to_gray(first))):
        out = colorizer.colorize(src)
        assert colorizer.last_info["ran"] is True
        assert out.shape == src.shape and out.dtype == np.uint8
        assert chroma_stats(out)[0] > GRAY_CHROMA, "output should have visible color"

def test_output_size_matches_input_for_odd_sizes():
    src = to_gray(cv2.resize(first, (333, 257)))
    assert colorizer.colorize(src).shape == src.shape

def test_switched_off_returns_input():
    c = Colorizer(ddcolor, enabled=False)
    src = to_gray(first)
    assert np.array_equal(c.colorize(src), src) and c.last_info["ran"] is False

def test_same_input_same_output():
    src = to_gray(first)
    assert np.array_equal(colorizer.colorize(src), colorizer.colorize(src))

def test_skip_rule_classification():
    for im in samples.values():
        assert classify_color(im) == "color"
        assert classify_color(to_gray(im)) == "gray"
        assert classify_color(to_sepia(to_gray(im))) == "sepia"

for t in [v for k, v in sorted(globals().items()) if k.startswith("test_")]:
    t(); print("PASS", t.__name__)

## 9. Resource smoke test

Logs peak GPU memory and seconds per image so the numbers can replace the plan in the design doc's GPU budget table (section 3.4).

In [ ]:
def smoke_test(sizes=((512, 512), (1024, 768), (2048, 1536)), repeats=3):
    rows = []
    for (w, h) in sizes:
        src = to_gray(cv2.resize(first, (w, h)))
        colorizer.colorize(src)                              # warm-up
        if DEVICE == "cuda":
            torch.cuda.reset_peak_memory_stats(); torch.cuda.synchronize()
        t0 = time.time()
        for _ in range(repeats):
            colorizer.colorize(src)
        if DEVICE == "cuda":
            torch.cuda.synchronize()
        rows.append(dict(size=f"{w}x{h}", sec_per_image=(time.time() - t0) / repeats,
                         peak_gpu_gb=(torch.cuda.max_memory_allocated() / 1e9) if DEVICE == "cuda" else float("nan")))
    return pd.DataFrame(rows)

smoke = smoke_test()
display(smoke.round(3))
smoke.to_csv(f"{OUT_DIR}/colorization_smoke_test.csv", index=False)

## 10. Findings and limits

*Fill in after running on the Colab T4.*

- Delta E and LPIPS versus the gray baseline: _TBD_
- Skip-rule accuracy and any threshold changes (faded color prints are the hard case): _TBD_
- Peak memory and seconds per image on a T4: _TBD_
- Failure cases (odd colors on uniforms, skin tones, cloudy skies): _TBD_
- Checkpoint comparison (`ddcolor_modelscope` vs `ddcolor_paper` vs `ddcolor_artistic`): _TBD_

**Known limits.** Colorization is a plausible guess, so it is labelled as one in the demo. DDColor sees a square resized input, so very large photos lose some color detail unless colorized at tile level.

**Code location:** `classify_color`, `is_grayscale`, `Colorizer` and the thresholds live in `src/colorization.py`; metrics come from `src/metrics.py`. `app/pipeline.py` calls the same classes.

## References

- Kang, X., Yang, T., Ouyang, W., et al. (2023). DDColor: Towards photo-realistic image colorization via dual decoders. *ICCV 2023*.
- Zhang, R., Isola, P., Efros, A. A., Shechtman, E., & Wang, O. (2018). The unreasonable effectiveness of deep features as a perceptual metric. *CVPR 2018*.
- Hasler, D., & Susstrunk, S. (2003). Measuring colourfulness in natural images. *SPIE Human Vision and Electronic Imaging VIII*.